# 05 · Branching Baselines

## Colab setup

In [ ]:
import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/neurosat"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, PROJECT_DIR)

print("Working folder:", os.getcwd())
print("Files:", os.listdir())

## Imports

In [ ]:
import pickle
import random
import time
import pandas as pd
import matplotlib.pyplot as plt

from cdcl import solve, vsids_literal, random_literal

## Settings

In [ ]:
SEED = 42
RL_TRAIN_SIZE = 2000
RL_TRAIN_MAX_VARS = 20
RL_VAL_SIZE = 100
EVAL_TEST_SIZE = 400
EVAL_LARGE_SIZE = 100
EVAL_LARGE_VARS = [40, 60]
MAX_DECISIONS = 5000

random.seed(SEED)

## Load data

In [ ]:
with open("train.pkl", "rb") as f:
    train_data = pickle.load(f)

with open("val.pkl", "rb") as f:
    val_data = pickle.load(f)

with open("test.pkl", "rb") as f:
    test_data = pickle.load(f)

with open("test_large.pkl", "rb") as f:
    large_test_data = pickle.load(f)

print("Train:", len(train_data))
print("Val:  ", len(val_data))
print("Test: ", len(test_data))

## Training formulas for reinforcement learning

In [ ]:
small_train = [p for p in train_data if p["n_vars"] <= RL_TRAIN_MAX_VARS]
rl_train = random.sample(small_train, RL_TRAIN_SIZE)

print("Small formulas available:", len(small_train))
print("Training formulas:       ", len(rl_train))
print("SAT share:               ", round(sum(p["label"] for p in rl_train) / len(rl_train), 2))

## Validation formulas

In [ ]:
small_val = [p for p in val_data if p["n_vars"] <= RL_TRAIN_MAX_VARS]
rl_val = random.sample(small_val, RL_VAL_SIZE)

print("Validation formulas:", len(rl_val))

## Evaluation formulas

In [ ]:
eval_sets = {"10–40": random.sample(test_data, EVAL_TEST_SIZE)}
for n in EVAL_LARGE_VARS:
    eval_sets[str(n)] = large_test_data[n][:EVAL_LARGE_SIZE]

for name, data in eval_sets.items():
    print(f"{name:6s}: {len(data)} formulas")

## Save datasets

In [ ]:
with open("rl_data.pkl", "wb") as f:
    pickle.dump({"train": rl_train, "val": rl_val, "eval": eval_sets}, f)

print("Saved rl_data.pkl")

## DLIS heuristic

In [ ]:
def dlis_literal(solver):
    counts = {}
    for clause in solver.clauses[:solver.n_original]:
        if any(solver.value(lit) is True for lit in clause):
            continue
        for lit in clause:
            if solver.value(lit) is None:
                counts[lit] = counts.get(lit, 0) + 1
    if not counts:
        return -solver.unassigned()[0]
    return max(counts, key=counts.get)


HEURISTICS = {
    "VSIDS": vsids_literal,
    "DLIS": dlis_literal,
    "Random": random_literal,
}

## Run function

In [ ]:
def run(problem, choose):
    start = time.perf_counter()
    solver = solve(problem["n_vars"], problem["clauses"], choose, MAX_DECISIONS)
    elapsed = time.perf_counter() - start

    return {
        "n_vars": problem["n_vars"],
        "label": problem["label"],
        "finished": solver.result is not None,
        "correct": solver.result == bool(problem["label"]),
        "decisions": solver.decisions,
        "conflicts": solver.conflicts,
        "learned": len(solver.lbds),
        "mean_lbd": sum(solver.lbds) / len(solver.lbds) if solver.lbds else 0,
        "lbds": solver.lbds,
        "ms": elapsed * 1000,
    }

## Run all heuristics

In [ ]:
rows = []

for set_name, data in eval_sets.items():
    for heuristic, choose in HEURISTICS.items():
        random.seed(SEED)
        for problem in data:
            row = run(problem, choose)
            row["set"] = set_name
            row["heuristic"] = heuristic
            rows.append(row)
        print(f"Done {set_name} with {heuristic}")

results = pd.DataFrame(rows)

## Correctness check

In [ ]:
finished = results[results["finished"]]

print("Runs:                 ", len(results))
print("Unfinished runs:      ", (~results["finished"]).sum())
print("Wrong finished answers:", (~finished["correct"]).sum())

## Mean results by heuristic and size

In [ ]:
summary = results.groupby(["set", "heuristic"])[["decisions", "conflicts", "mean_lbd", "ms"]].mean().round(2)
print(summary)

## Median decisions

In [ ]:
median_decisions = results.pivot_table(index="set", columns="heuristic", values="decisions", aggfunc="median")
print(median_decisions)

## SAT and UNSAT separately

In [ ]:
by_label = results.groupby(["set", "label", "heuristic"])[["decisions", "conflicts", "mean_lbd"]].mean().round(2)
print(by_label)

## Decisions plot

In [ ]:
mean_decisions = results.pivot_table(index="set", columns="heuristic", values="decisions", aggfunc="mean")
mean_decisions = mean_decisions.loc[list(eval_sets.keys()), list(HEURISTICS.keys())]

mean_decisions.plot(kind="bar", figsize=(8, 4), logy=True)
plt.xlabel("Variables")
plt.ylabel("Mean decisions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("baseline_decisions.png", dpi=150)
plt.show()

## Conflicts plot

In [ ]:
mean_conflicts = results.pivot_table(index="set", columns="heuristic", values="conflicts", aggfunc="mean")
mean_conflicts = mean_conflicts.loc[list(eval_sets.keys()), list(HEURISTICS.keys())]

mean_conflicts.plot(kind="bar", figsize=(8, 4), logy=True)
plt.xlabel("Variables")
plt.ylabel("Mean conflicts")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("baseline_conflicts.png", dpi=150)
plt.show()

## LBD distribution by heuristic

In [ ]:
plt.figure(figsize=(7, 4))

for heuristic in HEURISTICS:
    subset = results[(results["set"] == "10–40") & (results["heuristic"] == heuristic)]
    lbds = [lbd for row in subset["lbds"] for lbd in row]
    plt.hist(lbds, bins=range(1, 21), alpha=0.5, density=True, label=heuristic)

plt.xlabel("LBD of learned clause")
plt.ylabel("Share of learned clauses")
plt.legend()
plt.tight_layout()
plt.savefig("baseline_lbd_distribution.png", dpi=150)
plt.show()

## LBD and decisions

In [ ]:
subset = results[(results["set"] == "10–40") & (results["conflicts"] > 0)]

plt.figure(figsize=(7, 4))
for heuristic in HEURISTICS:
    h = subset[subset["heuristic"] == heuristic]
    plt.scatter(h["mean_lbd"], h["decisions"], s=10, alpha=0.5, label=heuristic)

plt.yscale("log")
plt.xlabel("Mean LBD of learned clauses")
plt.ylabel("Decisions")
plt.legend()
plt.tight_layout()
plt.savefig("baseline_lbd_vs_decisions.png", dpi=150)
plt.show()

for heuristic in HEURISTICS:
    h = subset[subset["heuristic"] == heuristic]
    print(f"{heuristic:6s} correlation between mean LBD and decisions: {h['mean_lbd'].corr(h['decisions'], method='spearman'):.2f}")

## Reference on validation formulas

In [ ]:
val_reference = {}

for heuristic, choose in HEURISTICS.items():
    random.seed(SEED)
    rows = [run(p, choose) for p in rl_val]
    val_reference[heuristic] = sum(r["decisions"] for r in rows) / len(rows)
    print(f"{heuristic:6s} mean decisions on validation formulas: {val_reference[heuristic]:.1f}")

## Save results

In [ ]:
with open("baselines.pkl", "wb") as f:
    pickle.dump({"results": results, "val_reference": val_reference}, f)

print("Saved baselines.pkl and the baseline plots")